In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNetCV
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error,mean_squared_error
pd.set_option('display.max_rows', None)
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import confusion_matrix,classification_report

In [2]:
df = pd.read_csv(r"C:\Users\cwech\Documents\March_Madness_Train_Model.csv")

In [3]:
df.head()

,year,team,year_team,seed,Finish,First_Rd,Second_Rd,Sweet_Sixteen,Elite_Eight,Final_Four,...,Seas_Turnovers,Seas_Fouls,Seas_Opp_PPG,Seas_Opp_3PT_Succ,Seas_Opp_3PT_Rate,Seas_Opp_Off_Rebound,Seas_Opp_Def_Rebound,Seas_Opp_Turnover,Seas_Opp_Fouls,Seas_Poss
0,2012,Kentucky,2012Kentucky,1,7,1,1,1,1,1,...,11.3,14.5,59.0,0.321,0.300,0.281,0.624,11.7,18.3,67.6
1,2012,Duke,2012Duke,2,1,0,0,0,0,0,...,12.1,18.2,68.5,0.317,0.239,0.314,0.665,12.8,20.8,69.9
2,2012,Baylor,2012Baylor,3,4,1,1,1,0,0,...,13.8,17.6,65.3,0.339,0.330,0.307,0.630,14.4,18.8,68.8
3,2012,Indiana,2012Indiana,4,3,1,1,0,0,0,...,12.6,17.6,65.5,0.349,0.317,0.287,0.665,13.2,20.7,68.3
4,2012,Wichita St,2012Wichita St,5,1,0,0,0,0,0,...,11.8,17.0,62.7,0.314,0.324,0.233,0.675,12.5,18.1,69.2


In [4]:
X = df.drop(['Finish','year','team','year_team','First_Rd','Second_Rd','Sweet_Sixteen','Elite_Eight',
             'Final_Four','Championship','Pts'],axis=1)
y_finish = df['Finish']
y_pts = df['Pts']

In [5]:
MM_finish_model = ElasticNetCV(l1_ratio=[.1,.5,.7,.9,.95,.99,1],n_alphas=1000,eps=0.001,max_iter=100000)
MM_pts_model = ElasticNetCV(l1_ratio=[.1,.5,.7,.9,.95,.99,1],n_alphas=1000,eps=0.001,max_iter=100000)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y_finish, test_size=0.25, random_state=101)

In [7]:
scaler = StandardScaler()

In [8]:
scaled_X_train = scaler.fit_transform(X_train)
scaled_X_test = scaler.transform(X_test)

In [9]:
MM_finish_model.fit(scaled_X_train,y_train)

ElasticNetCV(l1_ratio=[0.1, 0.5, 0.7, 0.9, 0.95, 0.99, 1], max_iter=100000,
             n_alphas=1000)

In [10]:
y_pred_finish = MM_finish_model.predict(scaled_X_test)

In [11]:
mean_absolute_error(y_test,y_pred_finish)

1.418448664288066

In [12]:
np.sqrt(mean_squared_error(y_test,y_pred_finish))

2.2256468251946617

In [13]:
MM_finish_model.get_params()

{'alphas': None,
 'copy_X': True,
 'cv': None,
 'eps': 0.001,
 'fit_intercept': True,
 'l1_ratio': [0.1, 0.5, 0.7, 0.9, 0.95, 0.99, 1],
 'max_iter': 100000,
 'n_alphas': 1000,
 'n_jobs': None,
 'positive': False,
 'precompute': 'auto',
 'random_state': None,
 'selection': 'cyclic',
 'tol': 0.0001,
 'verbose': 0}

In [14]:
coefs = pd.Series(index=X.columns,data=MM_finish_model.coef_)

In [15]:
coefs.sort_values()

Seas_3PT_Per           -1.082678
Seas_3PT_Rate          -0.983714
Seas_Poss              -0.194882
Seas_Opp_3PT_Succ      -0.192055
Seas_Fouls             -0.176242
AdjEM                  -0.163303
Seas_Opp_Fouls         -0.088409
Seas_Opp_Turnover      -0.023398
Seas_Opp_Def_Rebound   -0.000000
Seas_Opp_3PT_Rate      -0.000000
Seas_Opp_PPG           -0.000000
Seas_Turnovers          0.000000
seed                   -0.000000
Seas_FT_Succ            0.000000
AdjT                   -0.000000
AdjD                    0.000000
Seas_Def_Rebound_Per   -0.000000
Seas_Opp_Off_Rebound    0.003305
Seas_Off_Rebound_Per    0.016831
Luck                    0.195873
Seas_PPG                0.437225
AdjO                    0.508963
Seas_Succ_3PT           1.351972
dtype: float64

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y_pts, test_size=0.25, random_state=101)
scaled_X_train = scaler.fit_transform(X_train)
scaled_X_test = scaler.transform(X_test)
MM_pts_model.fit(scaled_X_train,y_train)

ElasticNetCV(l1_ratio=[0.1, 0.5, 0.7, 0.9, 0.95, 0.99, 1], max_iter=100000,
             n_alphas=1000)

In [17]:
y_pred_pts = MM_pts_model.predict(scaled_X_test)

In [18]:
mean_absolute_error(y_test,y_pred_pts)

3.8126099755080407

In [19]:
np.sqrt(mean_squared_error(y_test,y_pred_pts))

6.333646731345059

In [20]:
coefs = pd.Series(index=X.columns,data=MM_pts_model.coef_)
coefs.sort_values()

Seas_Fouls             -0.535744
Seas_Opp_Fouls         -0.432851
AdjD                   -0.391667
Seas_Opp_Def_Rebound   -0.349983
Seas_Def_Rebound_Per   -0.099140
Seas_Opp_3PT_Succ      -0.024295
Seas_Opp_3PT_Rate      -0.000087
seed                   -0.000000
Seas_Opp_PPG           -0.000000
Seas_Turnovers          0.000000
Seas_Poss               0.000000
Seas_3PT_Per            0.000000
Seas_Succ_3PT           0.000000
AdjT                    0.000000
Seas_3PT_Rate          -0.000000
Seas_Opp_Off_Rebound    0.030935
Seas_Opp_Turnover       0.264724
Seas_FT_Succ            0.332225
Seas_PPG                0.418852
Seas_Off_Rebound_Per    0.476926
AdjO                    0.622223
Luck                    1.202639
AdjEM                   2.551813
dtype: float64

In [21]:
pred_data = pd.read_csv(r"C:\Users\cwech\Downloads\March_Madness_2025_data.csv")

In [22]:
#pred_data = pred_data.set_index('team')
stats = pred_data.drop(['Finish','year','year_team','team','First_Rd','Second_Rd','Sweet_Sixteen','Elite_Eight',
             'Final_Four','Championship','Pts'],axis=1)

In [23]:
stats_scaled = scaler.transform(stats)

In [24]:
pred_finish = MM_finish_model.predict(stats_scaled)
pred_pts = MM_pts_model.predict(stats_scaled)
pred_finish = pd.Series(index=pred_data['team'],data=pred_finish)
pred_pts = pd.Series(index=pred_data['team'],data=pred_pts)

In [25]:
q = pd.concat([pred_finish,pred_pts],axis=1)

In [26]:
q = q.reset_index()
q = q.rename(columns={1: "pts_model",0: "finish_model"})

In [27]:
q = pred_data.merge(q, how='left', on=['team'])
q = q[['team','seed','finish_model','pts_model']]

In [28]:
#q.sort_values(by='finish_model',ascending=False)

In [29]:
log_model_First_Rd = LogisticRegressionCV(max_iter=5000, Cs=125, penalty='l2', solver='lbfgs')
log_model_Second_Rd = LogisticRegressionCV(max_iter=5000,Cs=128, penalty='l2', solver='liblinear')
log_model_SW16 = LogisticRegressionCV(max_iter=5000,Cs=125, penalty='l2', solver='lbfgs')
log_model_E8 = LogisticRegressionCV(max_iter=5000,Cs=135, penalty='l2', solver='lbfgs')
log_model_F4 = LogisticRegressionCV(max_iter=5000,Cs=135, penalty='l2', solver='lbfgs')
log_model_Champ = LogisticRegressionCV(max_iter=5000,Cs=135, penalty='l2', solver='lbfgs')

In [30]:
X = df.drop(['Finish','year','team','year_team','First_Rd','Second_Rd','Sweet_Sixteen','Elite_Eight',
             'Final_Four','Championship','Pts'],axis=1)
y_first_rd = df['First_Rd']
y_second_rd = df['Second_Rd']
y_sw16 = df['Sweet_Sixteen']
y_e8 = df['Elite_Eight']
y_f4 = df['Final_Four']
y_champ = df['Championship']

In [31]:
X.head()

,seed,AdjEM,AdjO,AdjD,AdjT,Luck,Seas_PPG,Seas_Succ_3PT,Seas_3PT_Per,Seas_FT_Succ,...,Seas_Turnovers,Seas_Fouls,Seas_Opp_PPG,Seas_Opp_3PT_Succ,Seas_Opp_3PT_Rate,Seas_Opp_Off_Rebound,Seas_Opp_Def_Rebound,Seas_Opp_Turnover,Seas_Opp_Fouls,Seas_Poss
0,1,32.59,122.2,89.6,65.4,0.032,76.7,0.224,0.371,0.720,...,11.3,14.5,59.0,0.321,0.300,0.281,0.624,11.7,18.3,67.6
1,2,19.70,117.7,98.0,66.4,0.057,77.5,0.314,0.376,0.702,...,12.1,18.2,68.5,0.317,0.239,0.314,0.665,12.8,20.8,69.9
2,3,21.29,116.0,94.7,65.6,0.059,74.3,0.272,0.376,0.752,...,13.8,17.6,65.3,0.339,0.330,0.307,0.630,14.4,18.8,68.8
3,4,22.55,120.2,97.6,66.7,0.007,77.3,0.248,0.433,0.762,...,12.6,17.6,65.5,0.349,0.317,0.287,0.665,13.2,20.7,68.3
4,5,22.36,116.1,93.8,65.7,-0.017,76.9,0.273,0.370,0.753,...,11.8,17.0,62.7,0.314,0.324,0.233,0.675,12.5,18.1,69.2


In [32]:
X_train, X_test, y_train, y_test = train_test_split(X, y_first_rd, test_size=0.25, random_state=101)
scaled_X_train = scaler.fit_transform(X_train)
scaled_X_test = scaler.transform(X_test)
log_model_First_Rd.fit(scaled_X_train,y_train)

C:\Users\cwech\anaconda3\Lib\site-packages\sklearn\model_selection\_split.py:700: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
# Set up parameter grid
#param_grid = {
#    'Cs': [135,145,140],
#    'solver': ['lbfgs']
#}

# Create grid search object
#grid_search = GridSearchCV(log_model_E8, param_grid, cv=5)

# Fit the grid search object to the data
#grid_search.fit(scaled_X_train, y_train)

# Print the best parameters
#print(grid_search.best_params_)

In [ ]:
coefs = pd.Series(index=X.columns,data=log_model_First_Rd.coef_[0])
coefs.sort_values()

In [ ]:
y_pred_frd = log_model_First_Rd.predict(scaled_X_test)

In [ ]:
print(classification_report(y_test,y_pred_frd))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y_second_rd, test_size=0.25, random_state=101)
scaled_X_train = scaler.fit_transform(X_train)
scaled_X_test = scaler.transform(X_test)
log_model_Second_Rd.fit(scaled_X_train,y_train)
y_pred_srd = log_model_Second_Rd.predict(scaled_X_test)
print(classification_report(y_test,y_pred_srd))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y_sw16, test_size=0.25, random_state=101)
scaled_X_train = scaler.fit_transform(X_train)
scaled_X_test = scaler.transform(X_test)
log_model_SW16.fit(scaled_X_train,y_train)
y_pred_sw16 = log_model_SW16.predict(scaled_X_test)
print(classification_report(y_test,y_pred_sw16))

In [ ]:
coefs = pd.Series(index=X.columns,data=log_model_SW16.coef_[0])
coefs.sort_values()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y_e8, test_size=0.25, random_state=101)
scaled_X_train = scaler.fit_transform(X_train)
scaled_X_test = scaler.transform(X_test)
log_model_E8.fit(scaled_X_train,y_train)
y_pred_e8 = log_model_E8.predict(scaled_X_test)
print(classification_report(y_test,y_pred_e8))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y_f4, test_size=0.25, random_state=101)
scaled_X_train = scaler.fit_transform(X_train)
scaled_X_test = scaler.transform(X_test)
log_model_F4.fit(scaled_X_train,y_train)
y_pred_f4 = log_model_F4.predict(scaled_X_test)
print(classification_report(y_test,y_pred_f4))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y_champ, test_size=0.25, random_state=101)
scaled_X_train = scaler.fit_transform(X_train)
scaled_X_test = scaler.transform(X_test)
log_model_Champ.fit(scaled_X_train,y_train)
y_pred_champ = log_model_Champ.predict(scaled_X_test)
print(classification_report(y_test,y_pred_champ))

In [ ]:
pred_frd = log_model_First_Rd.predict_proba(stats_scaled)
pred_srd = log_model_Second_Rd.predict_proba(stats_scaled)
pred_sw16 = log_model_SW16.predict_proba(stats_scaled)
pred_e8 = log_model_E8.predict_proba(stats_scaled)
pred_f4 = log_model_F4.predict_proba(stats_scaled)
pred_champ = log_model_Champ.predict_proba(stats_scaled)
pred_frd = pd.DataFrame(index=pred_data['team'],data=pred_frd)
pred_srd = pd.DataFrame(index=pred_data['team'],data=pred_srd)
pred_sw16 = pd.DataFrame(index=pred_data['team'],data=pred_sw16)
pred_e8 = pd.DataFrame(index=pred_data['team'],data=pred_e8)
pred_f4 = pd.DataFrame(index=pred_data['team'],data=pred_f4)
pred_champ = pd.DataFrame(index=pred_data['team'],data=pred_champ)

In [ ]:
pred_frd = pred_frd.reset_index()
pred_frd = pred_frd.drop(0,axis=1)
pred_frd = pred_frd.rename(columns={1: "First_Round"})
pred_srd = pred_srd.reset_index()
pred_srd = pred_srd.drop(0,axis=1)
pred_srd = pred_srd.rename(columns={1: "Second_Round"})
pred_sw16 = pred_sw16.reset_index()
pred_sw16 = pred_sw16.drop(0,axis=1)
pred_sw16 = pred_sw16.rename(columns={1: "Sweet_Sixteen"})
pred_e8 = pred_e8.reset_index()
pred_e8 = pred_e8.drop(0,axis=1)
pred_e8 = pred_e8.rename(columns={1: "Elite_Eight"})
pred_f4 = pred_f4.reset_index()
pred_f4 = pred_f4.drop(0,axis=1)
pred_f4 = pred_f4.rename(columns={1: "Final_Four"})
pred_champ = pred_champ.reset_index()
pred_champ = pred_champ.drop(0,axis=1)
pred_champ = pred_champ.rename(columns={1: "Champ"})

In [ ]:
q = pred_frd.merge(q, how='left', on=['team'])

In [ ]:
q = pred_srd.merge(q, how='left', on=['team'])
q = pred_sw16.merge(q, how='left', on=['team'])
q = pred_e8.merge(q, how='left', on=['team'])
q = pred_f4.merge(q, how='left', on=['team'])
q = pred_champ.merge(q, how='left', on=['team'])

In [28]:
X = df.drop(['Finish','year','team','year_team','First_Rd','Second_Rd','Sweet_Sixteen','Elite_Eight',
             'Final_Four','Championship','Pts'],axis=1)
y_finish = df['Finish']
X_train, X_test, y_train, y_test = train_test_split(X, y_finish, test_size=0.25, random_state=101)
scaled_X_train = scaler.fit_transform(X_train)
scaled_X_test = scaler.transform(X_test)
model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=500, learning_rate=0.01, max_depth=3,colsample_bytree=0.7,reg_alpha=1,reg_lambda=0,subsample=0.5)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("RMSE: %f" % (rmse))
pred_xgb = model.predict(stats_scaled)
pred_xgb = pd.DataFrame(index=pred_data['team'],data=pred_xgb)
pred_xgb = pred_xgb.reset_index()
pred_xgb = pred_xgb.rename(columns={0: "XGB"})
q = pred_xgb.merge(q, how='left', on=['team'])

RMSE: 1.606086


In [29]:
#params = {'objective': ['reg:squarederror', 'reg:linear', 'reg:gamma','reg:tweedie','binary:logistic','multi:softmax','multi:softprob']}
#grid_search = GridSearchCV(estimator=model, param_grid=params, cv=5, n_jobs=-1)
#grid_search.fit(X_train, y_train)
# Print the best hyperparameters and the corresponding score
#print("Best parameters: ", grid_search.best_params_)
#print("Best score: ", grid_search.best_score_)

In [30]:
#q = q[['team', 'seed','XGB','finish_model', 'pts_model', 'First_Round','Second_Round','Sweet_Sixteen','Elite_Eight','Final_Four','Champ']]
q = q[['team', 'seed','XGB','finish_model', 'pts_model']]
q = q.sort_values(by='finish_model',ascending=False)

In [31]:
q

,team,seed,XGB,finish_model,pts_model
67,Norfolk St,16,7.499398,9.779993,-1.740987
30,Gonzaga,8,7.352083,9.102244,7.511210
50,Yale,13,7.567477,8.883984,1.310004
4,Michigan St,2,7.299567,8.434358,8.132805
19,Memphis,5,9.162724,8.361672,6.021287
7,St John's,2,8.626774,7.964102,8.174120
25,Saint Mary's,7,8.764247,7.821922,7.573809
13,Arizona,4,9.095217,7.724756,7.786711
2,Houston,1,6.946177,7.399940,12.834254
36,New Mexico,10,9.048245,7.346691,3.083787


In [32]:
q.to_excel(r"C:\Users\cwech\Documents\Big_Dance_Results.xlsx",
             sheet_name='Results',index=False) 